# 02 - Clean monthly panel

This notebook converts the raw long-format Banco Central do Brasil SGS data downloaded in `01_download_bcb_sgs.ipynb` into a clean monthly wide panel.

The output panel has one row per month and one column per series. It will be the shared input for the descriptive plots and local projection estimates later in the project.

## Imports and paths

We load the core Python packages used throughout the project and import the transformation helpers from `src/transforms.py`. The path block is written so the notebook can run from either the project root or the `notebooks/` directory.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.transforms import (
    add_credit_shares,
    add_growth_rates,
    add_log_credit_variables,
    add_policy_variables,
    add_state_variables,
    ensure_datetime,
    first_last_nonmissing,
    missing_summary,
    monthly_panel_from_long,
)

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "bcb_sgs_all_long.csv"
DICTIONARY_FILE = PROJECT_ROOT / "data" / "series_dictionary.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "brazil_credit_monthly_panel.csv"

In [2]:

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [3]:

from src.transforms import (
    add_credit_shares,
    add_growth_rates,
    add_log_credit_variables,
    add_policy_variables,
    add_state_variables,
    ensure_datetime,
    first_last_nonmissing,
    missing_summary,
    monthly_panel_from_long,
)


In [4]:

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "bcb_sgs_all_long.csv"
DICTIONARY_FILE = PROJECT_ROOT / "data" / "series_dictionary.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "brazil_credit_monthly_panel.csv"

## Load raw data and series dictionary

The raw file is a tidy, long-format table with one observation per date-series pair. The series dictionary gives the manually verified SGS code, frequency, and project name for each series, which we use below when aggregating daily series to months.

In [5]:
raw = pd.read_csv(RAW_FILE)
series_dictionary = pd.read_csv(DICTIONARY_FILE)

display(raw.head())
display(series_dictionary.head())

print(f"Raw observations: {len(raw):,}")
print(f"Raw series in file: {raw['series'].nunique():,}")
print(f"Series in dictionary: {len(series_dictionary):,}")

,date,value,series
0,2011-01-01,10.85,selic_monthly_annualized
1,2011-02-01,11.17,selic_monthly_annualized
2,2011-03-01,11.62,selic_monthly_annualized
3,2011-04-01,11.74,selic_monthly_annualized
4,2011-05-01,11.92,selic_monthly_annualized


,name,series_id,source,frequency,unit,notes,verified
0,selic_target,432,BCB_SGS,monthly,percent_annual,Selic target rate; main monetary policy rate,no
1,selic_daily,11,BCB_SGS,daily,percent_daily,Interest rate - Selic; daily series; aggregate...,no
2,selic_annual_daily,1178,BCB_SGS,daily,percent_annual,Interest rate - Selic in annual terms (basis 2...,no
3,selic_monthly_annualized,4189,BCB_SGS,monthly,percent_annual,Interest rate - Selic accumulated in the month...,yes
4,inflation_target,13521,BCB_SGS,annual,percent_annual,Inflation target; useful for policy-rule or in...,yes


Raw observations: 6,510
Raw series in file: 35
Series in dictionary: 39


## Basic raw-data checks

Before reshaping, we verify that the expected long-format columns are present, parse the date column, and summarize coverage by series. These checks help catch failed downloads, malformed dates, or unexpected missing values before they become harder to diagnose in wide format.

In [6]:
required_columns = {"date", "value", "series"}
missing_columns = required_columns.difference(raw.columns)
if missing_columns:
    raise ValueError(f"Raw data is missing required columns: {sorted(missing_columns)}")

raw = ensure_datetime(raw, date_col="date")

print("Missing values by raw column:")
display(raw.isna().sum().rename("missing_count").to_frame())

date_range_by_series = (
    raw.groupby("series")
    .agg(first_date=("date", "min"), last_date=("date", "max"))
    .sort_index()
)
display(date_range_by_series)

observations_by_series = (
    raw.groupby("series")
    .size()
    .rename("observations")
    .sort_values(ascending=False)
    .to_frame()
)
display(observations_by_series)

Missing values by raw column:


,missing_count
date,0
value,177
series,0


,first_date,last_date
series,,
credit_total_stock,2011-01-01,2026-06-01
credit_total_stock_firms,2011-01-01,2026-06-01
credit_total_stock_households,2011-01-01,2026-06-01
credit_total_stock_to_gdp,2011-01-01,2026-06-01
directed_credit_stock,2011-01-01,2026-06-01
directed_credit_stock_firms,2011-01-01,2026-06-01
directed_credit_stock_households,2011-01-01,2026-06-01
directed_credit_stock_to_gdp,2011-01-01,2026-06-01
exchange_rate_usd_sale_avg,2011-01-01,2026-06-01


,observations
series,
credit_total_stock,186
credit_total_stock_firms,186
credit_total_stock_households,186
credit_total_stock_to_gdp,186
directed_credit_stock,186
directed_credit_stock_firms,186
directed_credit_stock_households,186
directed_credit_stock_to_gdp,186
exchange_rate_usd_sale_avg,186


## Convert to monthly panel

The empirical analysis is monthly, so every series needs to be aligned to a common monthly index. Daily series are converted to monthly averages, while monthly series keep the last observed value within each month. The result is a wide panel with one row per month and one column per project series name.

In [7]:
panel = monthly_panel_from_long(raw, dictionary_df=series_dictionary)

display(panel.head())
print(f"Monthly panel shape: {panel.shape[0]:,} rows x {panel.shape[1]:,} columns")

,month,credit_total_stock,credit_total_stock_firms,credit_total_stock_households,credit_total_stock_to_gdp,directed_credit_stock,directed_credit_stock_firms,directed_credit_stock_households,directed_credit_stock_to_gdp,exchange_rate_usd_sale_avg,...,interest_rate_new_nonrevolving_operations_firms,interest_rate_new_nonrevolving_operations_households,interest_rate_new_nonrevolving_operations_total,interest_rate_new_operations_firms,interest_rate_new_operations_households,interest_rate_new_operations_total,ipca,ipca_12m,selic_monthly_annualized,unemployment_rate_pnadc
0,2011-01-01,1718711.0,933467.0,785244.0,43.74,662040.0,434768.0,227272.0,16.85,1.6749,...,NaN,NaN,NaN,NaN,NaN,NaN,0.83,5.99,10.85,NaN
1,2011-02-01,1741509.0,947964.0,793544.0,43.78,670209.0,438131.0,232078.0,16.85,1.6680,...,NaN,NaN,NaN,NaN,NaN,NaN,0.80,6.01,11.17,NaN
2,2011-03-01,1759678.0,958884.0,800794.0,43.82,673815.0,436896.0,236919.0,16.78,1.6591,...,1.21,2.08,1.61,1.48,2.62,2.04,0.79,6.30,11.62,NaN
3,2011-04-01,1783487.0,971825.0,811662.0,43.94,680796.0,439588.0,241207.0,16.77,1.5864,...,1.25,2.09,1.64,1.52,2.66,2.07,0.77,6.51,11.74,NaN
4,2011-05-01,1812128.0,986470.0,825658.0,44.08,692755.0,445482.0,247273.0,16.85,1.6135,...,1.25,2.11,1.64,1.52,2.65,2.07,0.47,6.55,11.92,NaN


Monthly panel shape: 186 rows x 36 columns


## Construct key variables

This step creates the credit shares, accounting check variables, log credit stocks, monthly credit growth rates, policy-rate changes, optional exchange-rate changes, and the high-directed-share regime indicator used in the state-dependent specifications. The credit-stock identities are project assumptions, so they are built explicitly rather than hidden inside later estimation code.

In [8]:
panel = add_credit_shares(panel)
panel = add_log_credit_variables(panel)
panel = add_growth_rates(panel)
panel = add_policy_variables(panel)
panel = add_state_variables(panel)

constructed_columns = [
    "directed_credit_share",
    "free_credit_share",
    "credit_gap_check",
    "credit_gap_check_pct",
    "log_credit_total_stock",
    "log_free_credit_stock",
    "log_directed_credit_stock",
    "growth_credit_total_stock",
    "growth_free_credit_stock",
    "growth_directed_credit_stock",
    "delta_selic",
    "exchange_rate_log_change",
    "high_directed_share",
]
display(panel[[col for col in constructed_columns if col in panel.columns]].head())

,directed_credit_share,free_credit_share,credit_gap_check,credit_gap_check_pct,log_credit_total_stock,log_free_credit_stock,log_directed_credit_stock,growth_credit_total_stock,growth_free_credit_stock,growth_directed_credit_stock,delta_selic,exchange_rate_log_change,high_directed_share
0,0.385196,0.614804,0.0,0.000000e+00,14.357085,13.870634,13.403081,NaN,NaN,NaN,NaN,NaN,0.0
1,0.384844,0.615156,1.0,5.742147e-07,14.370263,13.884382,13.415345,1.317739,1.374853,1.226363,0.32,-0.412816,0.0
2,0.382919,0.617081,-1.0,-5.682858e-07,14.380641,13.897887,13.420711,1.037886,1.350405,0.536599,0.45,-0.535002,0.0
3,0.381722,0.618278,0.0,0.000000e+00,14.394081,13.913264,13.431018,1.343960,1.537757,1.030711,0.12,-4.480799,0.0
4,0.382288,0.617712,0.0,0.000000e+00,14.410012,13.928279,13.448432,1.593141,1.501515,1.741370,0.18,1.693843,0.0


## Diagnostics

These diagnostics are used to catch coding or concept mistakes before the cleaned panel is used in figures or local projections. In particular, the credit-gap check should be small if the total, free, and directed credit stock concepts line up as expected.

In [9]:
print(f"Panel date range: {panel['month'].min().date()} to {panel['month'].max().date()}")
print(f"Rows: {panel.shape[0]:,}")
print(f"Columns: {panel.shape[1]:,}")

print("Missing values by column:")
display(missing_summary(panel))

print("First and last non-missing date by column:")
display(first_last_nonmissing(panel, date_col="month"))

key_constructed_variables = [
    "directed_credit_share",
    "free_credit_share",
    "credit_gap_check_pct",
    "growth_credit_total_stock",
    "growth_free_credit_stock",
    "growth_directed_credit_stock",
    "delta_selic",
    "exchange_rate_log_change",
    "high_directed_share",
]
key_constructed_variables = [
    col for col in key_constructed_variables if col in panel.columns
]

print("Summary statistics for key constructed variables:")
display(panel[key_constructed_variables].describe().T)

print(
    "directed_credit_share min/max:",
    panel["directed_credit_share"].min(),
    panel["directed_credit_share"].max(),
)
print(
    "free_credit_share min/max:",
    panel["free_credit_share"].min(),
    panel["free_credit_share"].max(),
)
print(
    "credit_gap_check_pct mean absolute value:",
    panel["credit_gap_check_pct"].abs().mean(),
)
print(
    "credit_gap_check_pct max absolute value:",
    panel["credit_gap_check_pct"].abs().max(),
)

panel = panel.dropna(subset=["free_credit_stock", "directed_credit_stock"])

Panel date range: 2011-01-01 to 2026-06-01
Rows: 186
Columns: 50
Missing values by column:


,column,missing_count,missing_pct
0,month,0,0.000000
1,credit_total_stock,2,1.075269
2,credit_total_stock_firms,2,1.075269
3,credit_total_stock_households,2,1.075269
4,credit_total_stock_to_gdp,2,1.075269
5,directed_credit_stock,2,1.075269
6,directed_credit_stock_firms,2,1.075269
7,directed_credit_stock_households,2,1.075269
8,directed_credit_stock_to_gdp,2,1.075269
9,exchange_rate_usd_sale_avg,2,1.075269


First and last non-missing date by column:


,column,first_nonmissing,last_nonmissing
0,credit_total_stock,2011-01-01,2026-04-01
1,credit_total_stock_firms,2011-01-01,2026-04-01
2,credit_total_stock_households,2011-01-01,2026-04-01
3,credit_total_stock_to_gdp,2011-01-01,2026-04-01
4,directed_credit_stock,2011-01-01,2026-04-01
5,directed_credit_stock_firms,2011-01-01,2026-04-01
6,directed_credit_stock_households,2011-01-01,2026-04-01
7,directed_credit_stock_to_gdp,2011-01-01,2026-04-01
8,exchange_rate_usd_sale_avg,2011-01-01,2026-04-01
9,free_credit_stock,2011-01-01,2026-04-01


Summary statistics for key constructed variables:


,count,mean,std,min,25%,50%,75%,max
directed_credit_share,184.0,4.381310e-01,3.680076e-02,3.817219e-01,0.411931,0.424454,0.473541,5.038527e-01
free_credit_share,184.0,5.618690e-01,3.680075e-02,4.961473e-01,0.526459,0.575546,0.588069,6.182781e-01
credit_gap_check_pct,184.0,-1.726468e-08,1.600850e-07,-5.682858e-07,0.000000,0.000000,0.000000,5.742147e-07
growth_credit_total_stock,183.0,7.862151e-01,7.519727e-01,-1.021717e+00,0.269426,0.786694,1.343579,2.794261e+00
growth_free_credit_stock,183.0,7.419647e-01,9.265115e-01,-1.523022e+00,0.121257,0.733377,1.437435,4.303329e+00
growth_directed_credit_stock,183.0,8.501659e-01,8.811113e-01,-9.300987e-01,0.204259,0.817765,1.381024,3.310221e+00
delta_selic,185.0,1.918919e-02,3.661624e-01,-1.000000e+00,-0.190000,0.000000,0.190000,1.350000e+00
exchange_rate_log_change,183.0,6.012473e-01,3.471652e+00,-9.100995e+00,-1.696370,0.531744,2.561913,1.178393e+01
high_directed_share,184.0,5.000000e-01,5.013643e-01,0.000000e+00,0.000000,0.500000,1.000000,1.000000e+00


directed_credit_share min/max: 0.3817218740590764 0.5038526924331652
free_credit_share min/max: 0.4961473075668348 0.6182781259409236
credit_gap_check_pct mean absolute value: 7.196383051417783e-08
credit_gap_check_pct max absolute value: 5.742146609635666e-07


Added a line at the end of the cell above to trim the panel to the last observation period where we have the main variables directed and not directed credit. 

## Save cleaned panel

Finally, we save the cleaned monthly panel to `data/processed/`. This CSV is intentionally generated output: it can be recreated by rerunning the download and cleaning notebooks.

In [10]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_FILE, index=False)

print(f"Saved cleaned monthly panel to: {OUTPUT_FILE}")

Saved cleaned monthly panel to: c:\Users\chico\brazil-directed-credit-monetary-policy\data\processed\brazil_credit_monthly_panel.csv
